## NOVILLA Review 진위 분석 - 리뷰 수 증가 추이

In [6]:
import os
import sys
import io
from io import BytesIO
import csv
import google.auth
from google.cloud import bigquery
#from google.cloud import bigquery_storage

os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "../market-analysis-project-91130-5213911f50a5.json"
credentials, project_id = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
bqclient = bigquery.Client(credentials=credentials, project=project_id,)

# get the data from BigQuery
sql = f"""
select * from dw.novilla_mattress_npi_asin_reviews 
"""

df = bqclient.query(sql).to_dataframe()

In [11]:
df.head()

,asin,review_id,account_id,profile_name,star_rating,review_title,review_date,variation,verified_purchase,helpful_count,helpful_text,review_body,review_country,first_date
0,B0GH237V79,R2DITL2Y4BIA3X,AHJEHZXIWKJAGSNEEKNI2NXRN2TQ,Sarah H,4.0,Time will tell,2026-03-24,"Size: Twin | Style: 12""",1.0,0.0,None,"I purchased 2 of the 12"" twin size for my boys...",United States,2026-01-15
1,B0GH237V79,R26QL893MOTKU1,AFOY4PFJ2RXB2DC3EE3MGC74GZRA,Issa,5.0,extremely happy,2026-03-24,"Size: Twin | Style: 12""",1.0,0.0,None,I'm extremely happy with this mattress. I am s...,United States,2026-01-15
2,B0GH237V79,R1GSYVM30VMZPU,AE2LXK6ZN3QQ6F3YXSR66LPZZ67A,Baha,5.0,Very soft,2026-02-01,"Size: Twin | Style: 12""",1.0,0.0,None,"Very comfortable, soft, puffy , nice quality m...",United States,2026-01-15
3,B0GH237V79,R3N29VK0BQYG68,AF32VR5VOJ2RB7ZUGHPXXSVFGEJQ,Natallia,5.0,Highly recommend,2026-03-01,"Size: Twin | Style: 12""",1.0,0.0,None,Very comfortable mattress! It’s supportive but...,United States,2026-01-15
4,B0GH237V79,R1VSM1JB0M8OFL,AEV5EUCUBVAUISIJHE3Z4BWLU7KA,Saelys Linares,5.0,None,2026-03-02,"Size: Twin | Style: 12""",1.0,0.0,None,Estoy realmente muy encantada con este colchó...,United States,2026-01-15


In [13]:
"""
Novilla NPI ASIN: 출시일(first_date) 기준 누적 리뷰 곡선
- 의심 5개 ASIN (실선) + 대조군 2개 (점선)을 한 차트에 시각화
- 의심 ASIN의 'step-function' 누적 패턴 vs 대조군의 완만한 organic curve 대비
"""
import pandas as pd
import matplotlib.pyplot as plt

# ─── 설정 ─────────────────────────────────────────────
#CSV_PATH = "novilla_mattress_npi_asin_reviews_202606051530.csv"
OUT_PATH = "chart_cumulative_reviews.png"

# Windows: 'Malgun Gothic' / Mac: 'AppleGothic' / Linux: 'Noto Sans CJK KR'
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

# 색상·라벨 매핑 (color는 matplotlib tab palette 기준)
SUSPECT = {
    'B0GH2L8FWJ': '#d62728',
    'B0GH2HJ8V6': '#ff7f0e',
    'B0GH2M21J8': '#1f77b4',
    'B0GQH9F9PS': '#9467bd',
    'B0GJSKM5W4': '#8c564b',
}
CONTROL = {
    'B0GH2CXH2T': '#2ca02c',
    'B0GH2FT69Q': '#17becf',
}

# ─── 데이터 로드 ─────────────────────────────────────
#df = pd.read_csv(CSV_PATH)
df['review_date'] = pd.to_datetime(df['review_date'])
df['first_date'] = pd.to_datetime(df['first_date'])
df['days_since_launch'] = (df['review_date'] - df['first_date']).dt.days


def plot_cumulative(asins, ax, linestyle, linewidth, label_prefix):
    """ASIN 그룹별 누적 리뷰 곡선을 그리는 헬퍼"""
    for asin, color in asins.items():
        d = df[df['asin'] == asin].sort_values('review_date').copy()
        if d.empty:
            continue
        d['cumulative'] = range(1, len(d) + 1)
        ax.plot(d['days_since_launch'], d['cumulative'],
                color=color, linewidth=linewidth, linestyle=linestyle,
                label=f'{label_prefix}: {asin} ({len(d)}건)')


# ─── 차트 그리기 ─────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 7))

plot_cumulative(SUSPECT, ax, linestyle='-',  linewidth=2.5, label_prefix='의심')
plot_cumulative(CONTROL, ax, linestyle='--', linewidth=1.8, label_prefix='대조군')

# 출시 0~14일 organic 정상 구간 음영
ax.axvspan(0, 14, alpha=0.10, color='green',
           label='출시 0~14일 (정상 organic 첫 리뷰 구간)')

# 30 / 60 / 90일 reference line
for v, lbl in [(30, '30일'), (60, '60일'), (90, '90일')]:
    ax.axvline(v, color='gray', linestyle=':', alpha=0.5)
    ax.text(v, 2, f' {lbl}', fontsize=9, color='gray')

ax.set_xlabel('출시일(first_date)로부터 경과 일수')
ax.set_ylabel('누적 리뷰 수')
ax.set_title('Novilla NPI ASIN: 실제 출시일 기준 누적 리뷰 '
             '(의심 5개 = 침묵→burst, 대조군 2개 = 완만)',
             fontsize=13, fontweight='bold')
ax.legend(loc='lower right', fontsize=9.5)
ax.grid(alpha=0.3)
ax.set_xlim(-3, 150)

plt.tight_layout()
plt.savefig(OUT_PATH, dpi=130, bbox_inches='tight', facecolor='white')
plt.close()
print(f"Saved: {OUT_PATH}")

Saved: chart_cumulative_reviews.png
